In [1]:
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# Note on Load. Refer to the archived source for the detailed original comment.
classes_path = "/kaggle/input/datasets/fdiiiiii/d2-city-yolo/dataset/labels/classes.txt"
with open(classes_path, 'r') as f:
    class_names = [line.strip() for line in f.readlines()]

nc = len(class_names)
names = {i: name for i, name in enumerate(class_names)}

print(f"检测到 {nc} 个类别:")
for i, name in names.items():
    print(f"  {i}: {name}")

# 写入 d2-city.yaml
yaml_content = f"""
# D2-City dataset configuration
path: /kaggle/input/datasets/fdiiiiii/d2-city-yolo/dataset
train: /kaggle/input/datasets/fdiiiiii/d2-city-yolo/dataset/images/train
val: //kaggle/input/datasets/fdiiiiii/d2-city-yolo/dataset/images/val

nc: {nc}
names: {names}
"""

with open("/kaggle/working/d2-city.yaml", "w") as f:
    f.write(yaml_content)

print("已生成 /kaggle/working/d2-city.yaml")

检测到 12 个类别:
  0: car
  1: van
  2: bus
  3: truck
  4: person
  5: bicycle
  6: motorcycle
  7: open-tricycle
  8: closed-tricycle
  9: forklift
  10: large-block
  11: small-block
已生成 /kaggle/working/d2-city.yaml


In [3]:
%%writefile train_exp.py

# train_exp.py 

import os
import sys
from ultralytics import YOLO

"""
Part II: YOLO26s - 更换为更大模型并加载 COCO 预训练权重
"""

def experiment_1():
    """实验1: YOLO26s + pretrained (640px)"""
    model = YOLO("yolo26s.pt")          # 加载预训练权重
    
    model.train(
        data="./d2-city.yaml",
        epochs=100,
        cache=False,        # 关闭缓存
        imgsz=640,
        batch=10,          # 26s 显存占用更大，适当减小 batch
        amp=False,
        project=".",       # 保存根目录
        name="exp1"        # 子目录，最终保存在 runs_exp1/exp1
    )
    
    metrics = model.val()
    print(f"Exp1 mAP50-95: {metrics.box.map:.4f}")


"""
Part III: YOLO26n + 预训练权重 + 更高分辨率且数据增强
"""

def experiment_2():
    """实验2: YOLO26n + pretrained + 高分辨率960px + 数据增强"""
    model = YOLO("yolo26n.pt")
    
    model.train(
        data="./d2-city.yaml",
        epochs=100,
        cache=False,        # 关闭缓存
        imgsz=960,
        batch=10,
        rect=True,          # 矩形训练，减少填充
        mosaic=1.0,
        mixup=0.5,
        copy_paste=0.5,
        hsv_h=0.02,
        hsv_s=0.8,
        hsv_v=0.4,
        amp=False,
        project=".",
        name="exp2"
    )
    metrics = model.val()
    print(f"Exp2 mAP50-95: {metrics.box.map:.4f}")

if __name__ == "__main__":
    # Note on Experiment, Run, Parameters. Refer to the archived source for the detailed original comment.
    if len(sys.argv) > 1 and sys.argv[1] == "2":
        experiment_2() # 运行实验2
    else:
        experiment_1() # 运行实验1

Writing train_exp.py


In [4]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.5 MB/s eta 0:00:00


In [5]:
# Note on Experiment, Run. Refer to the archived source for the detailed original comment.
!python train_exp.py 2

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=10, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.5, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./d2-city.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, f

In [6]:
from ultralytics import YOLO
import os
import random

# Note on Model, Training, Load. Refer to the archived source for the detailed original comment.
model = YOLO("/kaggle/working/runs/detect/exp2/weights/best.pt")  # 或 last.pt

# Note on Testing, Image. Refer to the archived source for the detailed original comment.
test_img_dir = "/kaggle/input/datasets/xiaoyuwan/didi-detection/dataset/dataset_release/test"

# 随机选8个编号 (11000~11999)
indices = random.sample(range(11000, 12000), 8)
test_img_files = [os.path.join(test_img_dir, f"{idx}.jpg") for idx in indices]

# Note on Prediction. Refer to the archived source for the detailed original comment.
results = model(test_img_files)

# Note on Results. Refer to the archived source for the detailed original comment.
for file, result in zip(test_img_files, results):
    print(f"预测图像: {os.path.basename(file)}")
    result.show()


0: 960x960 6 cars, 10.8ms
1: 960x960 1 car, 10.8ms
2: 960x960 4 cars, 10.8ms
3: 960x960 9 cars, 2 buss, 10.8ms
4: 960x960 7 cars, 10.8ms
5: 960x960 2 cars, 2 vans, 10.8ms
6: 960x960 3 cars, 10.8ms
7: 960x960 7 cars, 1 bus, 10.8ms
Speed: 6.9ms preprocess, 10.8ms inference, 0.3ms postprocess per image at shape (1, 3, 960, 960)
预测图像: 11913.jpg


<PIL.Image.Image image mode=RGB size=1280x720>

预测图像: 11182.jpg


<PIL.Image.Image image mode=RGB size=1280x720>

预测图像: 11481.jpg


<PIL.Image.Image image mode=RGB size=1920x1080>

... (6 outputs left out)

预测图像: 11736.jpg


<PIL.Image.Image image mode=RGB size=1920x1080>

预测图像: 11336.jpg


<PIL.Image.Image image mode=RGB size=1920x1080>